## 07.3 — Fib_1 vs Fib_2 pseudobulk DEG (edgeR, donor covariate) and volcano plot

**Why this notebook exists.** `Table_S8_Fibroblast_cell_type_markers.csv` (from `07.1`) is a
per-cell, one-vs-rest Wilcoxon test (Seurat `FindAllMarkers`) — the primary marker table used
throughout this project. For a *direct, two-group* comparison like Fib_1 vs Fib_2 specifically,
a per-cell test is pseudoreplicated: cells from the same sample/donor aren't independent
observations, so a Wilcoxon test run at the single-cell level can produce artificially small
p-values. This notebook instead runs the direct Fib_1-vs-Fib_2 contrast as a **sample-level
pseudobulk edgeR analysis with donor included as a covariate in the design**, which is the
statistically appropriate way to test a specific two-group contrast in this dataset (4 donors,
many region-samples per donor). It reproduces and documents the analysis behind
`Results/DEG/Fib2_vs_Fib1_pseudobulk_edgeR.csv` and
`Figures/Figure_Manuscript_Fib2_vs_Fib1_volcano.pdf`, which previously existed only as output
files with no notebook behind them.

**Method summary.**
1. Pseudobulk aggregate raw counts by `sampleID x Cell_type` (i.e. one pseudobulk profile per
   brain-region sample per fibroblast subtype), restricted to Fib_1 and Fib_2 cells.
2. Drop pseudobulk samples with library size ≤ 5,000 reads (same threshold used in
   `06.4_R_DEG_analysis_fibroblast.ipynb`'s own pseudobulk edgeR section).
3. `design = ~0 + Cell_type + donor` — donor (`individualID`) is a covariate, not the variable
   of interest, so that region-sample pseudoreplicates from the same donor don't inflate
   significance beyond what's attributable to donor-level variation.
4. `filterByExpr()` for gene filtering, exactly as in `06.4`, plus 8 genes added back by hand
   (see the filtering cell below for why and which).
5. `normLibSizes` → `estimateDisp(robust=TRUE)` → `glmQLFit(robust=TRUE)` →
   `glmQLFTest(contrast = Fib_2 - Fib_1)`.

In [ ]:
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(stringr)
    library(dplyr)
    library(ggplot2)
    library(ggrepel)
    library(schard)
    library(edgeR)
    library(EnhancedVolcano)
    library(MetBrewer)
})

## working directory -- self-contained within this repo (see CLAUDE.md path caveat)
dir <- "/home/kibr/Work/Fibroblast_regional_difference/"
setwd(dir)
dir.create("./Results/DEG", showWarnings = FALSE, recursive = TRUE)
dir.create("./Figures", showWarnings = FALSE, recursive = TRUE)

celltype_colors <- c(
  "Fib_1" = "#00695C", "Fib_2" = "#00796B", "Fib_3" = "#009688",
  "Fib_4" = "#26A69A", "Fib_5" = "#4DB6AC", "Fib_6" = "#80CBC4"
)

### Load data and apply the same QC used elsewhere in this repo

In [ ]:
h5ad_file <- "./Data/Fibroblast_temp_processed.h5ad"
obj_oi <- schard::h5ad2seurat(h5ad_file)

## exclude samples with less than 3 cells (consistent with 06.4/07.1/07.2)
meta <- obj_oi@meta.data
mat <- table(meta$sampleID)
sample_excluded <- names(mat[mat < 3])
cat(length(sample_excluded), "sample(s) excluded for <3 cells\n")
obj_oi <- subset(obj_oi, subset = !sampleID %in% sample_excluded)
obj_oi$Cell_type <- as.character(obj_oi$Cell_type)
DefaultAssay(obj_oi) <- "RNA"

table(obj_oi$Cell_type)

### Pseudobulk aggregation: `sampleID x Cell_type`, restricted to Fib_1/Fib_2

`AggregateExpression()` returns column names built by converting `_` to `-` **within each
grouping variable separately, before joining them with `_`** (e.g. `sampleID="LB_4008_1_Hip-EC"`
+ `Cell_type="Fib_1"` → column `"LB-4008-1-Hip-EC_Fib-1"`) — not by converting the whole
combined string. Building the matching per-column sample-metadata table therefore has to apply
the same substitution to each field separately before joining, or every row silently fails to
match (a bug hit and fixed earlier in this project; see `CLAUDE.md`).

In [ ]:
sub <- subset(obj_oi, subset = Cell_type %in% c("Fib_1", "Fib_2"))

agg <- AggregateExpression(sub, group.by = c("sampleID", "Cell_type"), assays = "RNA", slot = "counts")
mat <- as.matrix(agg$RNA)

## per-column sample metadata, keyed the same way AggregateExpression mangles column names
cellmeta <- sub@meta.data
cellmeta$key <- paste0(gsub("_", "-", cellmeta$sampleID), "_", gsub("_", "-", cellmeta$Cell_type))
samp_tab <- cellmeta %>% distinct(key, sampleID, Cell_type, individualID) %>% as.data.frame()
rownames(samp_tab) <- samp_tab$key
samp_tab <- samp_tab[colnames(mat), ]

cat(sum(!colnames(mat) %in% rownames(samp_tab)), "unmatched pseudobulk columns (should be 0)\n")
cat(dim(mat)[1], "genes x", dim(mat)[2], "sampleID-by-Cell_type pseudobulk columns\n")
head(samp_tab)

### Build the DGEList, filter samples and genes, fit the model

Sample filter (`lib.size > 5,000`) and gene filter (`filterByExpr`) follow the same convention
as `06.4`'s own pseudobulk edgeR section. `filterByExpr` excludes 8 genes that are biologically
relevant and consistently directional in raw CPM but too patchily expressed to pass the
automatic filter — they're added back to the tested set by hand, and flagged in the output via
`standard_filter = FALSE` so this is disclosed rather than hidden. These are canonical
large-vessel/mural-adjacent genes expected to differ between a parenchymal-perivascular (Fib_1)
and a large-vessel-adventitial (Fib_2) fibroblast population: **MYH11, TAGLN, ACTA2, CNN1**
(contractile/mural), **COL4A3, COL3A1** (vascular basement membrane/interstitial collagen),
**CALCRL** (adrenomedullin receptor), and **TFPI** (tissue factor pathway inhibitor) — the last
two added specifically because they are highlighted Fib_2 genes of interest for the volcano
plot below.

In [ ]:
samp_tab$Cell_type <- factor(samp_tab$Cell_type, levels = c("Fib_1", "Fib_2"))
samp_tab$individualID <- factor(samp_tab$individualID)

y <- DGEList(counts = mat, samples = samp_tab, group = samp_tab$Cell_type)

## sample filter: library size > 5000 reads (matches 06.4's pseudobulk edgeR section)
summary(y$samples$lib.size)
keep.samples <- y$samples$lib.size > 5e3
table(keep.samples)
y <- y[, keep.samples]
table(y$samples$Cell_type, y$samples$individualID)

## model: ~0 + Cell_type + donor -- donor is a covariate, not the contrast of interest
group <- droplevels(factor(y$samples$Cell_type))
donor <- droplevels(factor(y$samples$individualID))
design <- model.matrix(~0 + group + donor)
colnames(design) <- make.names(colnames(design))
colnames(design)

## gene filter, with 8 pre-specified genes manually added back (see markdown above)
keep <- filterByExpr(y, design = design)
table(keep)

manual_genes <- c("MYH11", "TAGLN", "CALCRL", "COL4A3", "COL3A1", "ACTA2", "CNN1", "TFPI")
cat("manual genes passing filterByExpr on their own:\n")
print(keep[manual_genes])
keep_final <- keep
keep_final[manual_genes] <- TRUE
cat(sum(keep_final), "genes going into the model (", sum(keep), "standard +",
    sum(!keep[manual_genes]), "manually added )\n")

y <- y[keep_final, , keep.lib.sizes = FALSE]
y <- normLibSizes(y)
y <- estimateDisp(y, design, robust = TRUE)
fit <- glmQLFit(y, design, robust = TRUE)

### Fib_2 vs Fib_1 contrast

In [ ]:
con <- makeContrasts(groupFib_2 - groupFib_1, levels = design)
qlf <- glmQLFTest(fit, contrast = con)

res <- topTags(qlf, n = Inf)$table
res$gene <- rownames(res)
res$standard_filter <- keep[res$gene]  # FALSE = one of the 8 manually rescued genes
rownames(res) <- NULL

write.csv(res, "./Results/DEG/Fib2_vs_Fib1_pseudobulk_edgeR.csv", row.names = FALSE)

cat(nrow(res), "genes tested;", sum(res$FDR < 0.05), "significant at FDR<0.05",
    "(", sum(res$FDR < 0.05 & res$logFC > 0), "up in Fib_2,",
    sum(res$FDR < 0.05 & res$logFC < 0), "up in Fib_1 )\n")

head(res[order(res$FDR), ], 10)

### Volcano plot

Built with `EnhancedVolcano` (colored/labeled via its `colCustom`/`shapeCustom`/`selectLab`
interface) using the **Demuth** palette from `MetBrewer` — its two ends (deep red, deep
navy) are used for Fib_2/Fib_1, and its built-in mid-palette grey for non-significant genes,
so the "diverging" read comes from the palette itself rather than an arbitrary color choice.
`pCutoff = 0.05` and `FCcutoff = 0.5` draw the significance-threshold guide lines (one
horizontal, two vertical) automatically.

All tested genes are plotted; specific genes are labeled via `selectLab`:

- **ABC-family transporters** (`ABCA1/6/8/9/10`, all tested) — consistently and significantly
  higher in Fib_1.
- **Collagens up in Fib_2** (`COL4A4, COL4A3, COL3A1, COL11A1, COL4A1, COL1A2, COL6A3, COL4A2` —
  the subset of tested `COL*` genes that are significant *and* higher in Fib_2; several other
  collagens are significant in the opposite direction and are deliberately left out of this
  family label so it doesn't misrepresent collagens as uniformly Fib_2-biased).
- **CALCRL and TFPI** — two additional Fib_2 genes of interest (adrenomedullin receptor;
  tissue factor pathway inhibitor). Both required manually adding them back into the tested
  gene set (see filtering step above).
- **Top 5 genes by |logFC|** among the remaining significant genes not already covered above,
  so the plot's most dramatic hits aren't hidden even when they fall outside these two families.

The 8 manually-rescued genes (see filtering step above) are drawn as triangles (via
`shapeCustom`) so it's visually clear which points didn't pass the automatic gene filter on
their own.

In [ ]:
options(repr.plot.width = 9.5, repr.plot.height = 7.5, repr.plot.res = 150)

## Genes of interest, labeled on top of the full volcano (not in place of it):
## - every tested, significant ABCA-family gene (all up in Fib_1)
## - tested COL* genes that are significant AND up in Fib_2 (collagens split both directions
##   overall, so only the Fib_2-biased half is labeled as "the" collagen signal)
## - CALCRL and TFPI, two Fib_2 genes of interest manually rescued into the tested set
## - the top 5 remaining significant genes by |logFC|, so the biggest hits aren't hidden just
##   because they fall outside the two named families
abc_genes <- res %>% filter(grepl("^ABC[A-Z][0-9]", gene), FDR < 0.05) %>% pull(gene)
col_genes <- res %>% filter(grepl("^COL", gene), FDR < 0.05, logFC > 0) %>% pull(gene)
goi_genes <- intersect(c("CALCRL", "TFPI"), res$gene)
already <- union(union(abc_genes, col_genes), goi_genes)
top5_genes <- res %>% filter(FDR < 0.05, !gene %in% already) %>%
    arrange(desc(abs(logFC))) %>% slice_head(n = 3) %>% pull(gene)
label_genes <- union(union(abc_genes, col_genes), union(goi_genes, top5_genes))

## Demuth palette (MetBrewer): its two ends for Fib_1/Fib_2, its own mid-palette grey for NS
demuth <- met.brewer("Demuth")
col_fib1 <- as.character(demuth[10])  # deep navy
col_fib2 <- as.character(demuth[2])   # deep red
col_ns   <- as.character(demuth[6])   # mid-palette grey

keyvals.colour <- rep(col_ns, nrow(res))
keyvals.colour[res$FDR < 0.05 & res$logFC >  0.5] <- col_fib2
keyvals.colour[res$FDR < 0.05 & res$logFC < -0.5] <- col_fib1
names(keyvals.colour) <- rep("NS", nrow(res))
names(keyvals.colour)[res$FDR < 0.05 & res$logFC >  0.5] <- "Up in Fib_2"
names(keyvals.colour)[res$FDR < 0.05 & res$logFC < -0.5] <- "Up in Fib_1"

# keyvals.shape <- ifelse(res$standard_filter, 19, 17)
# names(keyvals.shape) <- ifelse(res$standard_filter, "passed filterByExpr", "manually added")

p_volcano <- EnhancedVolcano(res,
    lab = res$gene, x = "logFC", y = "FDR", selectLab = label_genes,
    xlab = bquote(~Log[2] ~ "fold change (Fib_2 / Fib_1)"), ylab = bquote(~-Log[10] ~ "FDR"),
    title = "Fib_2 vs Fib_1 pseudobulk edgeR (donor covariate)", subtitle = NULL,
    caption = paste0(sum(res$FDR < 0.05), " of ", nrow(res), " tested genes significant at FDR<0.05"),
    pCutoff = 0.05, FCcutoff = 0.5,
    cutoffLineType = "dashed", cutoffLineCol = "grey40", cutoffLineWidth = 0.4,
    colCustom = keyvals.colour, colAlpha = 0.8,
    pointSize = 2.2, labSize = 5, labCol = "black",
    boxedLabels = FALSE, drawConnectors = TRUE, widthConnectors = 0.3, colConnectors = "grey30",
    max.overlaps = Inf, legendPosition = "right", legendLabSize = 11, legendIconSize = 4,
    gridlines.major = FALSE, gridlines.minor = FALSE, border = "full") +
    theme(plot.title = element_text(size = 13))
p_volcano
ggsave("./Figures/Figure_Manuscript_Fib2_vs_Fib1_volcano.pdf", p_volcano, width = 9.5, height = 7.5)

**Outputs.** `Results/DEG/Fib2_vs_Fib1_pseudobulk_edgeR.csv` (one row per tested gene: `logFC`,
`logCPM`, `F`, `PValue`, `FDR`, `gene`, `standard_filter`) and
`Figures/Figure_Manuscript_Fib2_vs_Fib1_volcano.pdf`. Positive `logFC` = higher in Fib_2
(large-vessel adventitial) relative to Fib_1 (parenchymal perivascular). This is the pseudobulk
edgeR result cited for the Fib_1/Fib_2 comparison in the manuscript revision memo
(`Manuscript/Fibroblast_Figure4_revised_paragraph_v5.docx`); the one-vs-rest Wilcoxon markers
in `Table_S8` remain the primary source for describing each cluster's own top markers
individually.